In [0]:
CREATE OR REPLACE VIEW smart_meters.gold.vw_top_5_percent_consumers AS
WITH household_totals AS (
    -- 1. Calcula o consumo total de cada casa
    SELECT 
        f.dim_household_lclid AS lclid,
        h.acorn_group,
        SUM(f.energy_spent) AS total_kwh,
        SUM(f.money_spent) AS total_gbp
    FROM smart_meters.gold.fact_consumption f
    JOIN smart_meters.gold.dim_household h ON f.dim_household_lclid = h.lclid
    GROUP BY f.dim_household_lclid, h.acorn_group
),
ranked_households AS (
    -- 2. Atribui uma percentagem de ranking (0 a 1) dentro de cada grupo social
    SELECT 
        *,
        PERCENT_RANK() OVER (PARTITION BY acorn_group ORDER BY total_kwh DESC) AS pct_rank
    FROM household_totals
)
-- 3. Filtra apenas os que estão no top 5% (0.05)
SELECT 
    lclid,
    acorn_group,
    total_kwh,
    total_gbp
FROM ranked_households 
WHERE pct_rank <= 0.05
ORDER BY acorn_group, total_kwh DESC;

SELECT * FROM smart_meters.gold.vw_top_5_percent_consumers LIMIT 10